In [ ]:
%matplotlib inline

# Model Calibration / a.k.a. Parameter Estimation

Our models contain a lot of parameters that need to be set before the model can make useful predictions.
Some parameters in our models can be set directly from measurements, such as column lengths.
Others cannot be measured directly, such as dispersion coefficients, and need to be estimated by comparing simulation predictions to experimental data.

::::{grid} 1 2 2 2
:::{card}
:columns: 6

For our target `LRM` with `SMA` model, these are the parameters we can measure externally:
- `inlet.flow_rate`
- `inlet.c`
- `lrm.length`
- `lrm.diameter`
- `sma.capacity`
:::

:::{card}
:columns: 6
And these are the ones we need to fit:
- `lrm.axial_dispersion`
- `lrm.total_porosity`
- `sma.adsorption_rate`
- `sma.desorption_rate`
- `sma.characteristic_charge`
- `sma.steric_factor`

:::
::::

We need to start somewhere.

What is the chromatographic experiment with the least parameters you can think of?

The experiment we usually start with is a simple tracer pulse injection onto a chromatographic column with a non-binding molecule.

The only parameters that influence the elution are: `lrm.axial_dispersion` and `lrm.total_porosity`.

Your colleagues from the lab have shared their lab measurement with you in `./experimental_data/tracer_1.xlsx`. We can use the `pandas` library to import and plot the data.

## References

To quantify the difference between simulation and reference, **CADET-Process** provides a [`comparison`](https://cadet-process.readthedocs.io/en/latest/user_guide/process_evaluation/comparison.html) module.

To properly work with **CADET-Process**, the experimental data needs to be converted to an internal standard.
The `reference` module provides different classes for different types of experiments.
For in- and outgoing streams of unit operations, the `ReferenceIO` class must be used.

Similarly to the `SolutionIO` class, the `ReferenceIO` class also provides a plot method:

## Comparator

The `Comparator` class compares the simulation output with experimental data.
It provides several methods for visualizing and analyzing the differences between the data sets.
Users can choose from a range of metrics to quantify the differences between the two data sets, such as sum squared errors or shape comparison.

```{note}
It's also possible to add multiple references, e.g. for triplicate experiments or for different sensors.
```

## Difference Metrics
There are many metrics which can be used to quantify the difference between the simulation and the reference.
Most commonly, the sum squared error (SSE) is used.

However, SSE is often not an ideal measurement for chromatography.
Because of experimental non-idealities like pump delays and fluctuations in flow rate, there is a tendency for the peaks to shift in time.
This causes the optimizer to favor peak position over peak shape and can lead, for example, to an overestimation of axial dispersion.

In contrast, the peak shape is dictated by the physics of the physico-chemical interactions, while the position can shift slightly due to systematic errors like pump delays.
Hence, a metric which prioritizes the shape of the peaks being accurate over the peak eluting exactly at the correct time is preferable.
For this purpose, **CADET-Process** offers a `Shape` metric.

To add a difference metric, the following arguments need to be passed to the `add_difference_metric` method:
- `difference_metric`: The type of the metric.
- `reference`: The reference which should be used for the metric.
- `solution_path`: The path to the corresponding solution in the simulation results.

## Reference Model

Now, we need our prepared model to compare to the experimental data.

::::{grid} 1 1 2 2

:::{grid-item}
:columns: 8

For the injection, we need to introduce two sections:
- In the first section, which lasts $1~\text{s}$, the concentration of the tracer inlet is $1.0~\text{mM}$,
- afterwards it is $0.0~\text{mM}$.

The flow rate is a constant $1~\text{mL}~\text{min}^{-1}$.
:::

:::{grid-item}
:columns: 4

```{image} ./resources/dextran_inlet.png
:width: 60%
:align: center
```
:::
::::

The difference can also be visualized:

And we can calculate the exact value of the difference:

The comparison shows that there is still a large discrepancy between simulation and experiment.

## Optimization
To find the porosity with the best agreement between simulation and data, we can screen some porosities and compare them to our data:

### Visualization

In [ ]:
%matplotlib ipympl

from ipywidgets import interact, interactive
import ipywidgets as widgets
import matplotlib.pyplot as plt
from scipy.interpolate import PchipInterpolator
from CADETProcess import plotting

try:
    ui.close()
except Exception:
    pass
try:
    plt.close(fig)
except Exception:
    pass


sim_res = scan_simulation_results[porosities[0]]

reference_interpolated = PchipInterpolator(reference.time, reference.solution[:, 0])(sim_res.time_complete)

fig, (ax, ax_score) = plotting.setup_figure(
    ncols=2,
    layout="1.5_col",
    scale_with_subplots=True,
)

ax_score.bar(
    porosities,
    metrics.values(),
    color="red",
    width=porosities[1]-porosities[0],
    alpha=0.6,
)
vline = ax_score.axvline(x=0.2, linestyle=":", color="grey")
ax_score.set_ylabel("SSE [-]")
ax_score.set_xlabel("porosity [-]")

fig.tight_layout()

# Visualization
def graph_column(porosity=0.4):
    ax.clear()
    sim_res = scan_simulation_results[porosity]
    ax.fill_between(
        sim_res.time_complete,
        reference_interpolated,
        sim_res.solution.outlet.outlet.solution[:, 0],
        color="red",
        alpha=0.6
    )
    line_sim = ax.plot(sim_res.time_complete, sim_res.solution.outlet.outlet.solution)[0]
    line_ref = ax.plot(reference.time, reference.solution, ":", color="black")
    vline.set_xdata([porosity, porosity])
    ax.set_ylim(-0.004, 0.113)
    ax.set_xlim(-5, 105)

style = {'description_width': 'initial'}
_ = interact(graph_column, porosity=widgets.SelectionSlider(layout={'width': '800px'}, style=style, description='porosity', options = porosities))

Instead of manually adjusting these parameters, an `OptimizationProblem` can be set up which automatically determines the parameter values.
For this purpose, an `OptimizationProblem` is defined and the process is added as an evaluation object.

Then, the optimization variables are added.
Note that the parameter path associates the variable with the parameter of the corresponding column unit operation.

Before the difference metrics, which we want to minimize, the `Process` needs to be simulated.
For this purpose, register the `Cadet` simulator instance as an evaluator.

Now, when adding the `Comparator` (which determines the difference metrics) as objective function, the simulator can be added to the `required` list.
Note that the number of metrics needs to be passed as `n_objectives`.

## Callbacks
A `callback` function is a user function that is called periodically by the optimizer in order to allow the user to query the state of the optimization.
For example, a simple user callback function might be used to plot results.
The function is called after each iteration for all best individuals at that state.

The callback signature may include any of the following arguments:
- `results`: obj

    x or final result of evaluation toolchain.
- `individual`: {class}`Individual`, optional

    Information about current step of optimizer.
- `evaluation_object`: obj, optional

    Current evaluation object.
- `callbacks_dir`: Path, optional

    Path to store results.

## Optimizer

A couple of optimizers are available in **CADET-Process**.
Depending on the problem at hand, some optimizers might outperform others.
Generally, `U_NSGA3`, a genetic algorithm, is a robust choice.
While not necessarily the most efficient, it usually manages to handle complex problems with multiple dimensions, constraints, and objectives.
Here, we limit the number of cores, the population size, as well as the maximum number of generations.

### Optimization Progress and Results

The `OptimizationResults` which are returned contain information about the progress of the optimization.
For example, the attributes `x` and `f` contain the final value(s) of parameters and the objective function.

After optimization, several figures can be plotted to visualize the results.
For example, the convergence plot shows how the function value changes with the number of evaluations.

The `plot_objectives` method shows the objective function values of all evaluated individuals.
Here, lighter colors represent later evaluations.
Note that by default the values are plotted on a log scale if they span many orders of magnitude.
To disable this, set `autoscale=False`.

Note that more figures are created for constrained optimization, as well as multi-objective optimization.
All figures are also saved automatically in the `working_directory`.
Moreover, results are stored in a `.csv` file.
- The `results_all.csv` file contains information about all evaluated individuals.
- The `results_last.csv` file contains information about the last generation of evaluated individuals.
- The `results_pareto.csv` file contains only the best individual(s).

We can also look at the callbacks that were generated in the results folder.